# Lecture 5 (03 Oct 2026) — Five real tasks with NumPy and an LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kosavina/hse-python-2026/blob/main/Lecture_5.ipynb)

Every task follows the same five steps:

```
 1 DISCUSS  ──►  2 PROMPT  ──►  3 VERIFY  ──►  4 FEATURE  ──►  5 VERIFY AGAIN
  by hand        the model        check cell      the model        check cell
                                     │                                 │
                                     └───── failed? FIX with the model ┘
```

1. **Discuss.** How would you do it by hand? We work out a small case on the board. Those numbers go into the check cell **before** any code exists.
2. **Prompt.** Write the prompt in the box, send it to the model, and paste the code into the empty cell. The spec lists the names the check cell expects; put them in your prompt.
3. **Verify.** Run the check cell (don't edit it). Then answer the reading questions: they are about **shape, dtype and axis**.
4. **Feature.** Paste your working code back into the chat and ask for the change. Ask the model to *change* the code rather than start again.
5. **Verify again.** The second check cell.

**If a check fails:** paste the failed `assert` line and its message into the chat, ask the model to **explain which line causes it first**, and only then to fix it. Rerun **all** the check cells of the task, not just the one that failed.

**Prompt rules:** give the input shapes; ask it to print the shape of every intermediate array; name the axis in words; state the conventions (rates, fees, units); ask for vectorised NumPy with no Python loops over the data.

AI level: Full AI (AIAS 4). Fill in the AI log at the end.

## Setup — NumPy and the SBER data

In [1]:
# Setup. Run this first. Tasks 4 and 5 use the SBER arrays; Tasks 1-3 only need numpy.
import os, urllib.request
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/kosavina/hse-python-2026/main/data/SBER_2025_daily.csv"
DATA_FILE = "SBER_2025_daily.csv"
for candidate in (DATA_FILE, os.path.join("data", DATA_FILE), os.path.join("..", "data", DATA_FILE)):
    if os.path.exists(candidate):
        DATA_FILE = candidate
        break
else:
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

# Columns in the file: TRADEDATE, OPEN, HIGH, LOW, CLOSE, VOLUME (shares), VALUE (rubles)
dates = np.loadtxt(DATA_FILE, delimiter=",", skiprows=1, usecols=0, dtype=str)
open_, high, low, close, volume, value = np.loadtxt(
    DATA_FILE, delimiter=",", skiprows=1, usecols=(1, 2, 3, 4, 5, 6), unpack=True)

print("dates", dates.shape, dates.dtype, "| close", close.shape, close.dtype)
print(dates[0], close[0], "...", dates[-1], close[-1])
assert dates.shape == close.shape == value.shape == (254,)

dates (254,) <U10 | close (254,) float64
2025-01-03 272.25 ... 2025-12-30 299.9


---
## Task 1 — Russian income tax on a progressive scale

**The story.** Since 1 January 2025 salary income is taxed on a five-step scale. A rumour says: *"earn over 2.4 M a year and your whole income is taxed at 15 %"*. Is it true? And why does a 300 000 ₽ salary pay less, after tax, from September?

| Annual income, ₽ | Rate on the part inside the bracket |
|---|---|
| up to 2 400 000 | 13 % |
| 2 400 000 – 5 000 000 | 15 % |
| 5 000 000 – 20 000 000 | 18 % |
| 20 000 000 – 50 000 000 | 20 % |
| above 50 000 000 | 22 % |

### 1.1 Discuss — by hand
1. Tax on 3 000 000 ₽? What is the **effective** rate (tax / income)? So, is the rumour true?
2. Tax on exactly 2.4 M, 5 M, 20 M and 50 M. (Each one builds on the one before.)
3. Describe the calculation as a rule a machine could follow, for **every bracket at once**. What is the shape of the intermediate table for `n` incomes?

In [ ]:
HAND_1 = "..."   # your hand answers

**The spec (what the check cell expects):** a function `ndfl(income)` that takes a 1-D array of annual incomes and returns an array of taxes with the **same shape**.

### 1.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 1.3 Verify *(don't edit)*

In [ ]:
inc = np.array([1e6, 2.4e6, 3e6, 5e6, 20e6, 50e6])
assert ndfl(inc).shape == inc.shape, "one tax per income, same shape"
assert np.allclose(ndfl(inc), [130_000, 312_000, 402_000, 702_000, 3_402_000, 9_402_000]), ndfl(inc)
assert np.isclose(ndfl(np.array([0.0]))[0], 0), "no income, no tax"
t = ndfl(np.linspace(0, 70e6, 701))
assert (np.diff(t) >= 0).all(), "tax must never fall when income rises"
eff = ndfl(inc) / inc
assert (np.diff(eff) >= 0).all() and eff.max() < 0.22, "the effective rate rises, but stays below the top rate"
print("OK: 3 M RUB pays", f"{ndfl(np.array([3e6]))[0]:,.0f}", "RUB, effective", f"{ndfl(np.array([3e6]))[0] / 3e6:.2%}")

**Read the code.**
1. Where are the bracket bounds? What is the upper bound of the top bracket, and why?
2. Which axis of the intermediate array is "brackets"? Which line sums over it?
3. For an income of 1 M, what does the clipping do to the 15 % … 22 % brackets?

### 1.4 Feature — tax withheld month by month
The employer withholds tax every month on the income **since January** (*нарастающим итогом*): this month's tax = `ndfl(income to date) − ndfl(income to last month)`.

**The spec:** `monthly_tax(monthly_salary)` for 12 equal salaries returns two arrays, `tax` and `net`, both of shape `(12,)`.

*Before you run:* with 300 000 ₽ a month, in which month does the 2.4 M threshold bite, and what is the take-home pay before and after? And with 500 000 ₽ a month, what is withheld in **May**?

### 1.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
tax, net = monthly_tax(300_000)
assert tax.shape == net.shape == (12,)
assert np.allclose(tax[:8], 39_000) and np.allclose(tax[8:], 45_000), tax
assert np.allclose(tax + net, 300_000)
assert np.isclose(tax.sum(), ndfl(np.array([3.6e6]))[0]), "the months must add up to the year's tax"
tax5, _ = monthly_tax(500_000)
assert np.isclose(tax5[4], 67_000), "May: 400 000 at 13 % + 100 000 at 15 %"
assert np.allclose(monthly_tax(150_000)[0], 19_500), "150 000 a month never crosses 2.4 M"
print("OK: from September, 300 000 a month nets", f"{net[8]:,.0f}", "instead of", f"{net[0]:,.0f}")

---
## Task 2 — Cross rates from the Central Bank table

**The story.** The Bank of Russia publishes how many rubles one unit of each currency costs. You want every pair — EUR/USD, USD/CNY, USD/JPY — and the real cost of changing money in a bank.

The data cell below holds the official rates set for **30 December 2025**. Look at the `nominal` column: the rate is **per 10 lira, per 100 tenge, per 100 yen**.

### 2.1 Discuss — by hand
1. How many dollars is one euro? How many yuan is one dollar?
2. How many yen is one dollar? What do you get if you forget the nominal, and how would you notice?
3. Write the rule for "units of currency *j* for 1 unit of currency *i*". What three properties must the full table have?

In [ ]:
# CBR official rates set for 30.12.2025 (cbr.ru, "Официальные курсы валют на заданную дату").
# Careful: `rate` is rubles per `nominal` units -- per 10 lira, per 100 tenge, per 100 yen.
codes   = ["USD", "EUR", "CNY", "GBP", "TRY", "KZT", "JPY"]
nominal = np.array([1, 1, 1, 1, 10, 100, 100])
rate    = np.array([77.4466, 91.4775, 11.0501, 104.6071, 18.0933, 15.3481, 49.5436])

In [ ]:
HAND_2 = "..."   # your hand answers

**The spec:** `rub`, the rubles for **one** unit of each currency, with shape `(7,)`; and `M`, with shape `(7, 7)`, where `M[i, j]` is how many units of `codes[j]` you get for one unit of `codes[i]`.

### 2.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 2.3 Verify *(don't edit)*

In [ ]:
assert rub.shape == (7,) and M.shape == (7, 7)
assert np.isclose(rub[codes.index("JPY")], 0.495436), "rubles per ONE yen -- was the nominal used?"
assert np.allclose(np.diag(M), 1), "a currency against itself is 1"
assert np.allclose(M * M.T, 1), "USD->EUR times EUR->USD must be 1"
assert np.allclose(M[:, :, None] * M[None, :, :], M[:, None, :]), "i->j->k must equal i->k"
i = codes.index
assert np.isclose(M[i("EUR"), i("USD")], 1.1812, atol=1e-4), M[i("EUR"), i("USD")]
assert np.isclose(M[i("USD"), i("CNY")], 7.0087, atol=1e-4), M[i("USD"), i("CNY")]
assert np.isclose(M[i("USD"), i("JPY")], 156.32, atol=1e-2), M[i("USD"), i("JPY")]
print("OK: EUR/USD", round(M[i("EUR"), i("USD")], 4), " USD/JPY", round(M[i("USD"), i("JPY")], 2))

**Read the code.**
1. Which line divides a column by a row? Write the two shapes and the result's shape.
2. If `M` were transposed, which asserts would still pass, and which one would catch it?
3. The triangle check multiplies a `(7, 7, 1)` array by a `(1, 7, 7)` array. What is the shape of the result, and what does element `[i, j, k]` mean?

### 2.4 Feature — the bank's spread, and a travel budget
A bank **sells** you currency at the official rate × (1 + s) and **buys** it back at the official rate × (1 − s), with s = 2 %.

**The spec:**
- `round_trip(rub_amount, code, s)` converts rubles into the currency and straight back, and returns the rubles you end with.
- `amounts` is an array of shape `(7,)` aligned with `codes`: a trip needs 500 USD, 300 EUR, 2 000 CNY and 50 000 JPY.
- `cost_cbr` and `cost_bank` are the trip's cost in rubles at the official rate and at the bank's selling rate, computed with `@`.

*Before you run:* how many rubles are left of 100 000 after a round trip? Does the answer depend on the currency?

### 2.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
for code_ in codes:
    assert np.isclose(round_trip(100_000, code_, 0.02), 100_000 * 0.98 / 1.02), code_
assert np.isclose(round_trip(100_000, "USD", 0.0), 100_000), "no spread, no loss"
assert amounts.shape == (7,), "one amount per currency, aligned with codes"
assert np.isclose(cost_cbr, 500 * 77.4466 + 300 * 91.4775 + 2000 * 11.0501 + 50_000 * 0.495436)
assert np.isclose(cost_bank, cost_cbr * 1.02)
print(f"OK: round trip keeps {100_000 * 0.98 / 1.02:,.2f} of 100 000 RUB; trip costs {cost_bank:,.2f} RUB at the bank")

---
## Task 3 — What a bond is worth, and what yield its price implies

**The story.** The Bank of Russia key rate is 16 % (since 19 December 2025). A bond that pays a 10 % coupon must then trade **below** its face value. How far below? And if the market quotes a price, what yield does that price imply?

**The bond:** face value 1 000 ₽, an annual coupon of 10 % (100 ₽ a year), 3 years to maturity. Its price at yield *y* is the sum of the cash flows, each discounted: $$\sum \frac{CF_t}{(1 + y)^t}.$$

### 3.1 Discuss — by hand
1. List the cash flows and their times.
2. Price at y = 12 %? (Three divisions and a sum.)
3. Price at y = 10 %? Why exactly that number?
4. A zero-coupon bond (no coupons, 1 000 at the end) at 12 %?
5. Arrays: what shape are the cash flows, the discount factors for *k* yields at once, and the prices?

In [ ]:
HAND_3 = "..."   # your hand answers

**The spec:** `bond_price(face, coupon_rate, years, y)`, where `y` is a number **or** a 1-D array of yields; the price has the same shape as `y`.

### 3.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 3.3 Verify *(don't edit)*

In [ ]:
ys = np.array([0.08, 0.10, 0.12, 0.14, 0.16, 0.20])
p = bond_price(1000, 0.10, 3, ys)
assert p.shape == ys.shape, "one price per yield"
assert np.allclose(p, [1051.54, 1000.00, 951.96, 907.13, 865.25, 789.35], atol=0.01), p
assert np.isclose(bond_price(1000, 0.10, 3, 0.10), 1000), "yield = coupon -> price = face"
assert np.isclose(bond_price(1000, 0.0, 3, 0.12), 1000 / 1.12**3), "zero-coupon closed form"
g = bond_price(1000, 0.10, 3, np.linspace(0.01, 0.30, 30))
assert (np.diff(g) < 0).all() and (np.diff(g, 2) > 0).all(), "price falls with yield, and the curve is convex"
print("OK: at 12 % the bond is worth", round(float(bond_price(1000, 0.10, 3, 0.12)), 2))

**Read the code.**
1. Which axis is summed? What would summing over the other axis give?
2. Does the time array start at 1 or at 0? Which assert would fail if it started at 0?
3. What does `np.diff(g, 2) > 0` check, in words?

### 3.4 Feature — the yield from a market price
The market quotes the bond at **950 ₽**. What yield to maturity does that imply?

**The spec:** `ytm(price, face, coupon_rate, years)` returns `(yield, price_error)`. Do it the NumPy way: price the bond on a fine grid of yields from 0 % to 50 %, and take the yield whose price is closest (`argmin`).

*Before you run:* between which two yields from the table above must the answer lie?

### 3.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert np.isclose(ytm(1000, 1000, 0.10, 3)[0], 0.10, atol=1e-5), "a par bond yields its coupon"
assert np.isclose(ytm(951.96, 1000, 0.10, 3)[0], 0.12, atol=1e-4), "round trip with the table above"
y950 = ytm(950, 1000, 0.10, 3)[0]
assert 0.12 < y950 < 0.14, "950 is between the 12 % and 14 % prices"
assert np.isclose(bond_price(1000, 0.10, 3, y950), 950, atol=0.01), "plug the yield back in"
assert 0.08 < ytm(1050, 1000, 0.10, 3)[0] < 0.10, "a premium bond yields less than its coupon"
print(f"OK: at 950 the yield to maturity is {y950:.2%}")

---
## Task 4 — The average price of SBER, done properly

**The story.** A broker's report says: *"you bought at 305, below the average price for the month"*. Which average? The standard benchmark is the **VWAP**, the volume-weighted average price: the rubles traded divided by the shares traded.

The setup cell loaded `dates`, `open_`, `high`, `low`, `close`, `volume` (shares) and `value` (rubles), each of shape `(254,)`. If you overwrote any of them, rerun the setup cell.

### 4.1 Discuss — by hand
1. The VWAP for 3 January 2025 from the first row of the file: VALUE = 11 853 565 984.9 ₽, VOLUME = 43 086 870 shares. Between which two numbers of that row must it lie?
2. The VWAP for the **whole year**: which of the two is right, the total rubles over the total shares, or the plain mean of the 254 daily VWAPs? Why?
3. **PREDICT:** which of the two will come out larger for 2025?

In [ ]:
HAND_4 = "..."   # your hand answers

**The spec:** `vwap`, the daily VWAP with shape `(254,)`; `vwap_year`, one number for the year; `turnover_bn`, the total rubles traded in billions.

### 4.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 4.3 Verify *(don't edit)*

In [ ]:
assert vwap.shape == (254,), vwap.shape
assert np.isclose(vwap[0], 275.11, atol=0.01), "3 Jan by calculator: 11 853 565 984.9 / 43 086 870"
assert ((low <= vwap) & (vwap <= high)).all(), "an average price must lie inside the day's range"
assert np.isclose(vwap_year, value.sum() / volume.sum()), "yearly VWAP = total rubles / total shares"
assert not np.isclose(vwap_year, vwap.mean()), "the yearly VWAP is not the plain mean of the daily ones"
assert np.isclose(turnover_bn, value.sum() / 1e9)
print(f"OK: yearly VWAP {vwap_year:.2f} RUB, turnover {turnover_bn:,.1f} bn RUB")

**Read the code.**
1. What is the dtype of `vwap`? What would it be if `value` had been read as text?
2. If `volume` and `value` were swapped, what would a "price" look like, and which assert would catch it?
3. Which days pull the plain mean away from the yearly VWAP: busy ones or quiet ones?

### 4.4 Feature — a monthly table, and the busiest days
**The spec:**
- `months`, the 12 months as strings (`"2025-01"`, …), with shape `(12,)`;
- for each month, arrays of shape `(12,)`: `days` (trading days), `turnover` (bn ₽), `vol_m` (shares), `vwap_m` (the month's rubles over the month's shares), `mean_daily_m` (the plain mean of the daily VWAPs) and `last_close_m`;
- `top5_dates`, the 5 days with the largest turnover, busiest first. Print their daily returns too.

*Before you run:* how many trading days does November have? (Look at the calendar of 2025 in the file.)

### 4.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
assert len(months) == 12 and days.sum() == 254
assert days[months == "2025-11"][0] == 20, "November includes the working Saturday, 1 Nov"
assert np.isclose(turnover.sum(), value.sum() / 1e9), "the months must add up to the year"
assert np.isclose((vwap_m * vol_m).sum() / vol_m.sum(), vwap_year), "volume-weighted months = the year"
month_of_day_ = np.array([d[:7] for d in dates])
for m_, v_ in zip(months, vwap_m):
    sel = month_of_day_ == m_
    assert low[sel].min() <= v_ <= high[sel].max(), m_
assert len(top5_dates) == 5 and top5_dates[0] == "2025-02-12", top5_dates
print("OK: busiest day", top5_dates[0], "| widest gap VWAP vs mean-of-days:",
      months[np.abs(vwap_m - mean_daily_m).argmax()])

---
## Task 5 — A moving-average rule against buy-and-hold

**The story.** A classic rule: *hold SBER while its 20-day average price is above its 50-day average; otherwise hold cash*. Would it have beaten simply holding the stock in 2025?

Uses `close` and `dates` from the setup cell.

### 5.1 Discuss — by hand
1. The 3-day moving average of `[10, 11, 12, 13, 14]`. How long is it, and to which **day** does each value belong?
2. On which day of the file do both the 20-day and the 50-day averages first exist?
3. The rule is checked at the close of day *t*. Over which day's return can it first earn? Why not the same day's?
4. What is a fair "buy-and-hold" to compare with: from 3 January, or from the first day of the rule?
5. A fee of 0.05 % on every change of position: how does it change the total return, as a formula?

In [ ]:
HAND_5 = "..."   # your hand answers

**The spec (the check cell calls these):**
- `sma(x, n)` returns the moving average of shape `(len(x) − n + 1,)`, where element `k` is the mean of `x[k:k+n]`.
- `run(close, dates, fast=20, slow=50)` returns a **dict** with these keys:
  - `"dates_bt"`: the dates from the first day of the rule;
  - `"r"`: the daily returns over that window;
  - `"pos"`: the position (1 or 0) held over each of those returns;
  - `"strat"` and `"bh"`: the total returns of the rule and of buy-and-hold;
  - `"n_changes"`: the number of position changes, counting the first entry;
  - `"in_market"`: the share of days in the stock.

### 5.2 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

### 5.3 Verify *(don't edit)*

In [ ]:
assert np.allclose(sma(np.array([10, 11, 12, 13, 14.0]), 3), [11, 12, 13]), "hand example"
assert len(sma(close, 20)) == 254 - 20 + 1 and np.isclose(sma(close, 20)[0], close[:20].mean())
a = run(close, dates)
assert a["dates_bt"][0] == "2025-03-14", "the backtest starts on the first day both averages exist"
assert a["pos"].shape == a["r"].shape, "one position per daily return"
assert np.isclose(np.prod(1 + a["r"]) - 1, a["bh"]), "always in the market must equal buy-and-hold"
assert np.isclose(a["bh"], -0.0643, atol=1e-4), "buy-and-hold over the same window"

# No look-ahead: a price shock from day k on must not change any decision taken before day k.
k = int(np.nonzero(dates == "2025-05-15")[0][0])
close_shock = close.copy()
close_shock[k:] *= 2
b = run(close_shock, dates)
j = k - (50 - 1)                     # day k in backtest positions
assert np.array_equal(a["pos"][:j], b["pos"][:j]), "a decision used prices from the future"

assert np.isclose(a["strat"], -0.1124, atol=1e-4) and a["n_changes"] == 9, (a["strat"], a["n_changes"])
print(f"OK: strategy {a['strat']:+.2%} vs buy-and-hold {a['bh']:+.2%}")

**Read the code.**
1. Find the line that pairs a signal with a return. Point to the exact slices, and say which day each one belongs to.
2. The "price shock" check doubles every price from 15 May on. Why must the positions **before** 15 May stay exactly the same? What would it mean if they changed?
3. Why is buy-and-hold here negative, when SBER rose over 2025?

### 5.4 Feature — fees, and a second pair of windows
**The spec:** `run(close, dates, fast=20, slow=50, fee=0.0)` also returns `"strat_net"`, the rule's return after a fee of `fee` on every day the position changes, including the first entry. Then compare SMA 20/50 with SMA 10/30.

*Before you run:* using the formula from 5.1 and the numbers above, what should the 20/50 return after a 0.05 % fee be?

### 5.5 Your prompt
*(write the prompt you sent to the model here)*

In [ ]:
# Paste the model's code here, then run the cell.

In [ ]:
res = run(close, dates, 20, 50, fee=0.0005)
assert np.isclose(1 + res["strat_net"], (1 + res["strat"]) * (1 - 0.0005) ** res["n_changes"]), "fees multiply"
assert np.isclose(run(close, dates, 20, 50, fee=0.0)["strat_net"], res["strat"]), "zero fee changes nothing"
r2 = run(close, dates, 10, 30)
assert r2["dates_bt"][0] == "2025-02-14", "the window starts on day `slow`"
assert np.isclose(r2["bh"], -0.0322, atol=1e-4), r2["bh"]
print(f"OK: SMA20/50 after fees {res['strat_net']:+.2%}; SMA10/30 {r2['strat']:+.2%} vs buy-and-hold {r2['bh']:+.2%}")

---
## AI log

Required by HSE AI-disclosure rules, and by this course regardless.

| Task | Model used | Prompts until the check passed | What did you have to tell it the second time? |
|---|---|---|---|
| 1 NDFL | | | |
| 1 Feature | | | |
| 2 Cross rates | | | |
| 2 Feature | | | |
| 3 Bond | | | |
| 3 Feature | | | |
| 4 VWAP | | | |
| 4 Feature | | | |
| 5 SMA strategy | | | |
| 5 Feature | | | |
